In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON


In [2]:
def run(endpoint, query):
    s = SPARQLWrapper(endpoint)
    s.setMethod("POST")
    s.setTimeout(60)
    s.setReturnFormat(JSON)
    s.addParameter("format", "json")
    s.addCustomHttpHeader("User-Agent", "NL2SPARQL/0.1 (jeffry.cacho@rwth-aachen.de)")
    s.setQuery(query)
    return s.query().convert()["results"]["bindings"]


In [3]:
DBLP = "https://sparql.dblp.org/sparql"

q ="""
PREFIX dblp: <https://dblp.org/rdf/schema#>
SELECT ?pub ?title ?year WHERE {
  ?pub a dblp:Publication ; dblp:title ?title ; dblp:yearOfPublication ?year .
  FILTER CONTAINS(LCASE(STR(?title)), 'attention is all you need')
  FILTER(?year = "2017"^^<http://www.w3.org/2001/XMLSchema#gYear>)
}
LIMIT 10
"""

run(endpoint=DBLP, query=q)

[{'pub': {'type': 'uri',
   'value': 'https://dblp.org/rec/conf/nips/VaswaniSPUJGKP17'},
  'title': {'type': 'literal', 'value': 'Attention is All you Need.'},
  'year': {'datatype': 'http://www.w3.org/2001/XMLSchema#gYear',
   'type': 'literal',
   'value': '2017'}},
 {'pub': {'type': 'uri',
   'value': 'https://dblp.org/rec/journals/corr/VaswaniSPUJGKP17'},
  'title': {'type': 'literal', 'value': 'Attention Is All You Need.'},
  'year': {'datatype': 'http://www.w3.org/2001/XMLSchema#gYear',
   'type': 'literal',
   'value': '2017'}}]

In [4]:
WD = "https://query.wikidata.org/sparql"

q = """
PREFIX wd: <http://www.wikidata.org/entity/>
SELECT ?paper WHERE { VALUES ?paper { wd:Q30249683 } }"""

q = """
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX bd: <http://www.bigdata.com/rdf#>
PREFIX mwapi: <https://www.mediawiki.org/ontology#API/>

SELECT ?paper WHERE {
  SERVICE wikibase:mwapi {
    bd:serviceParam wikibase:endpoint "www.wikidata.org" .
    bd:serviceParam wikibase:api "EntitySearch" .
    bd:serviceParam mwapi:search "Attention Is All You Need" .
    bd:serviceParam mwapi:language "en" .
    ?paper wikibase:apiOutputItem mwapi:item .
  }
}
LIMIT 5
"""

run(endpoint=WD, query=q)

[{'paper': {'type': 'uri',
   'value': 'http://www.wikidata.org/entity/Q30249683'}}]

In [2]:
from sparqlx import SPARQLWrapper

async def run(endpoint, query):
    wrapper = SPARQLWrapper(endpoint, aclient_config={"timeout": 60, "headers": {
        "User-Agent": "NL2SPARQL/0.1 (https://github.com/yourusername/nl2sparql)"
    }})
    result = await wrapper.aquery(query)
    return result.json()["results"]["bindings"]

# Usage

q="""
PREFIX dblp: <https://dblp.org/rdf/schema#>

SELECT DISTINCT ?s ?p ?label (STRLEN(STR(?label)) AS ?labelLen)
       (IF(LCASE(STR(?label))=LCASE("Raquel Urtasun"), 1, 0) AS ?exact)
WHERE {
  VALUES ?cls { <https://dblp.org/rdf/schema#Creator> }
  VALUES ?p { dblp:creatorName dblp:orcid dblp:creatorNote dblp:primaryAffiliation dblp:primaryCreatorName dblp:affiliation }
  ?s a ?cls ; ?p ?label .
  FILTER(isLiteral(?label))
  FILTER(CONTAINS(LCASE(STR(?label)), LCASE("Raquel Urtasun")))
}
LIMIT 30"""


result = await run("https://sparql.dblp.org/sparql", q)


In [3]:
result

[{'s': {'type': 'uri', 'value': 'https://dblp.org/pid/u/RaquelUrtasun'},
  'p': {'type': 'uri', 'value': 'https://dblp.org/rdf/schema#creatorName'},
  'label': {'type': 'literal', 'value': 'Raquel Urtasun'},
  'labelLen': {'datatype': 'http://www.w3.org/2001/XMLSchema#int',
   'type': 'literal',
   'value': '14'},
  'exact': {'datatype': 'http://www.w3.org/2001/XMLSchema#int',
   'type': 'literal',
   'value': '1'}},
 {'s': {'type': 'uri', 'value': 'https://dblp.org/pid/u/RaquelUrtasun'},
  'p': {'type': 'uri', 'value': 'https://dblp.org/rdf/schema#creatorName'},
  'label': {'type': 'literal', 'value': 'Raquel Urtasun Waabi'},
  'labelLen': {'datatype': 'http://www.w3.org/2001/XMLSchema#int',
   'type': 'literal',
   'value': '20'},
  'exact': {'datatype': 'http://www.w3.org/2001/XMLSchema#int',
   'type': 'literal',
   'value': '0'}},
 {'s': {'type': 'uri', 'value': 'https://dblp.org/pid/u/RaquelUrtasun'},
  'p': {'type': 'uri',
   'value': 'https://dblp.org/rdf/schema#primaryCreatorN